In [26]:
!pip install numpy
!pip install pandas


In [51]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [52]:
credits_df = pd.read_csv('tmdb_5000_credits.csv')
movies_df = pd.read_csv('tmdb_5000_movies.csv')


In [59]:

merged_df = movies_df.merge(credits_df, on='title')


In [57]:
print(credits_df.columns)
print(movies_df.columns)

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')


In [63]:
features = ['title', 'overview', 'genres', 'keywords', 'cast', 'crew']
for feature in features:
    merged_df[feature] = merged_df[feature].fillna('')


In [65]:
def combine_features(row):
    return row['title'] + ' ' + row['overview'] + ' ' + row['genres'] + ' ' + row['keywords'] + ' ' + row['cast'] + ' ' + row['crew']

merged_df['combined_features'] = merged_df.apply(combine_features, axis=1)


In [67]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(merged_df['combined_features'])


In [69]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [71]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = merged_df[merged_df['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]  # Get top 10 similar movies
    movie_indices = [i[0] for i in sim_scores]
    return merged_df['title'].iloc[movie_indices]

# Example usage
print(get_recommendations('The Dark Knight'))


3                    The Dark Knight Rises
119                          Batman Begins
9       Batman v Superman: Dawn of Justice
281                      American Gangster
287                       Django Unchained
72                           Suicide Squad
1849                            GoodFellas
30                            Spider-Man 2
1162                    The Social Network
693                              Gone Girl
Name: title, dtype: object


In [79]:
# Example usage
print(get_recommendations('Harry Potter and the Half-Blood Prince'))


113     Harry Potter and the Order of the Phoenix
114           Harry Potter and the Goblet of Fire
276       Harry Potter and the Chamber of Secrets
197      Harry Potter and the Philosopher's Stone
191      Harry Potter and the Prisoner of Azkaban
9              Batman v Superman: Dawn of Justice
1849                                   GoodFellas
322                             The Fifth Element
72                                  Suicide Squad
365                                       Contact
Name: title, dtype: object
